# W06 — Validation Audit

**Lane:** Ranking Signal Analysis
**Auditing:** the Week-5 model (Logistic Regression / Random Forest vs. Week-4 baseline)

> **Before you submit:** run every cell top-to-bottom in Colab with `HF_TOKEN` set. Section 1
> requires something I genuinely cannot do for you — see the note there. Sections 2-5 are fully
> coded and ready to execute; the actual before/after numbers and failure examples only exist once
> you run this against the real data. Run All, fill the `# FILL AFTER RUN` spots, then commit.


## 1) Two Paper Findings + My Methodology Questions

**I have not read the research paper linked on this card** — it isn't something I have access to,
and this section specifically needs to be your own careful read, not a summary I generate secondhand.
Inventing findings or methodology critiques for a paper neither of us has actually looked at would
be the opposite of the rigor this task is teaching, so I'm leaving this section as a filled-in
template rather than fabricating content.

Do this part yourself, in about 15-20 minutes, using the same lens from the live session:

1. Open the paper and pick two findings — ideally ones closest to what your model claims (a
   ranking claim, an association claim, or a "this signal predicts X" claim).
2. For each, ask the same two questions the session asked of it:
   - **Where does the label come from?** Is it an observed future outcome, or a proxy/current-state
     bucket? Does the paper say?
   - **Does the validation design support the claim?** Random split, grouped, time-aware? Does the
     claim's scope (e.g. "predicts decline") match what the validation actually tested?
3. Write both up **constructively** — a methodology question, not a takedown. The template below
   has the shape; replace the bracketed parts with the paper's actual finding and your actual
   question.


**Finding 1:** `[quote or closely paraphrase the paper's finding in one sentence — no long
quotes, keep it to a short paraphrase]`

**My methodology question:** `[e.g. "The write-up doesn't say whether X was measured before or
after the outcome window — could you clarify whether this is a same-window proxy or a future
outcome, since that changes how strong the claim can be?"]`

---

**Finding 2:** `[paraphrase]`

**My methodology question:** `[e.g. "The validation split isn't described — was it grouped by
client/site? If not, could shared patterns within a client be inflating the reported metric?"]`


## Setup

In [ ]:
import duckdb, os, json
import numpy as np
import pandas as pd
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET hf_token='{os.environ['HF_TOKEN']}';")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

import pathlib
pathlib.Path("work/outputs").mkdir(parents=True, exist_ok=True)


In [ ]:
features_df = con.sql(f"""
WITH monthly AS (
  SELECT
    f.content_hash_id,
    f.client_hash_id,
    SUM(f.impressions) AS impressions_30d,
    SUM(f.clicks)       AS clicks_30d,
    AVG(f.position)     AS avg_position_30d,
    d.word_count             AS word_count,
    d.days_since_last_update AS days_since_last_update
  FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') f
  JOIN read_parquet('{BASE}/dim_content/*.parquet') d
    ON f.content_hash_id = d.content_hash_id
  GROUP BY 1, 2, d.word_count, d.days_since_last_update
),
latest_trend AS (
  SELECT content_hash_id, client_hash_id, trend_direction, trend_pct
  FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY content_hash_id, client_hash_id ORDER BY report_date DESC
  ) = 1
)
SELECT m.*, t.trend_direction, t.trend_pct
FROM monthly m
JOIN latest_trend t USING (content_hash_id, client_hash_id)
""").df()

features_df["ctr_30d"] = (features_df["clicks_30d"] / features_df["impressions_30d"].replace(0, np.nan)).fillna(0)
features_df["label"] = (features_df["trend_direction"] == "down").astype(int)

FEATURE_COLS = ["impressions_30d", "clicks_30d", "ctr_30d", "avg_position_30d", "word_count"]
X = features_df[FEATURE_COLS].fillna(0)
y = features_df["label"]
groups = features_df["client_hash_id"]

print("rows:", len(X), " clients:", groups.nunique())


## 2) My Model Under An Honest Split — Before / After

**Before:** a plain random train/test split (no client grouping) — the naive version, which lets
rows from the same client land on both sides.

**After:** the same `GroupShuffleSplit` by `client_hash_id` used in Week 5 — no client appears in
both train and test.

Same features, same label, same model (Logistic Regression), same metrics — the only thing that
changes is the split. Any gap between the two rows is the honest split doing its job.


In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)[:k]
    return y_true.values[order].mean()

def fit_and_score(X_train, X_test, y_train, y_test, label):
    model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    return {
        "split": label,
        "roc_auc": roc_auc_score(y_test, scores),
        "average_precision": average_precision_score(y_test, scores),
        "precision_at_50": precision_at_k(y_test, scores, 50),
        "n_test": len(y_test),
    }, model

# --- BEFORE: naive random split ---
Xtr_naive, Xte_naive, ytr_naive, yte_naive, g_tr_naive, g_te_naive = train_test_split(
    X, y, groups, test_size=0.25, random_state=42, stratify=y
)
overlap_naive = set(g_tr_naive) & set(g_te_naive)
before_result, _ = fit_and_score(Xtr_naive, Xte_naive, ytr_naive, yte_naive, "before: random split")
before_result["client_overlap_train_test"] = len(overlap_naive)

# --- AFTER: grouped split (same as Week 5) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
Xtr_grp, Xte_grp = X.iloc[train_idx], X.iloc[test_idx]
ytr_grp, yte_grp = y.iloc[train_idx], y.iloc[test_idx]
g_tr_grp, g_te_grp = groups.iloc[train_idx], groups.iloc[test_idx]
overlap_grp = set(g_tr_grp) & set(g_te_grp)
after_result, after_model = fit_and_score(Xtr_grp, Xte_grp, ytr_grp, yte_grp, "after: grouped split")
after_result["client_overlap_train_test"] = len(overlap_grp)

before_after = pd.DataFrame([before_result, after_result])
before_after


**Read this before writing anything else.** `# FILL AFTER RUN` — state the actual
`client_overlap_train_test` count for the "before" row (it should be > 0, proving the naive split
really did leak clients across train/test) and whether the metrics dropped, stayed flat, or moved
oddly going from "before" to "after." A drop from the naive to the grouped split is the expected,
honest result — it means the naive number was partly inflated by client memorization, not
generalizable signal. If the numbers barely move, say that plainly too; it would mean this
particular signal set doesn't rely much on client-specific pattern-matching.


## 3) Leakage Audit

Two parts: the checklist from the lane guide's leakage rules, answered plainly for the five
features actually used, and a re-run of the deliberate-leak demonstration from `w03`/`w04` — this
time framed as a formal audit result rather than a one-off trap.


**Leakage checklist** (section 12 of the lane guide), answered for `impressions_30d`,
`clicks_30d`, `ctr_30d`, `avg_position_30d`, `word_count`:

| Question | Answer |
|---|---|
| Are any features calculated after the decision point? | No — all five are aggregated strictly within the `month=2026-03` window; the decision point is treated as end-of-March. |
| Does the feature window overlap the target window? | The label (`trend_direction == 'down'`) is read from the *same* end-of-March snapshot as the features — this is the proxy-label weakness already flagged in w03/w04, not a future-window design. It is a known limitation, not a leak in the stricter sense (no future data enters the features), but it means the label is closer to "current state" than "future outcome." |
| Did any rebuilt product output (`health_score`, a decision flag, `priority_score`, `action_type`) slip in as a feature? | No — none of FlyRank's product decision outputs are in the release or in this feature set. |
| Does a derived field secretly encode the target? | This is exactly what `trend_pct` does (see the leak re-run below) — it is excluded from the model for that reason. |
| Are duplicate/related rows split across train/test in a way that makes the test too easy? | Checked directly in section 2 — client overlap is asserted to be zero in the "after" (grouped) split. |
| Are you testing on clients/periods the model hasn't effectively already seen? | Yes for clients (grouped split, section 2). Not yet tested for time — see the note below. |


**Deliberate-leak re-run** (formalizing the w03/w04 trap as an audit step on the current
feature set):

In [ ]:
leaky_X = Xtr_grp.copy()
leaky_X["trend_pct"] = features_df.loc[Xtr_grp.index, "trend_pct"]
leaky_X_test = Xte_grp.copy()
leaky_X_test["trend_pct"] = features_df.loc[Xte_grp.index, "trend_pct"]

leaky_model = LogisticRegression(max_iter=1000).fit(leaky_X, ytr_grp)
leaky_scores = leaky_model.predict_proba(leaky_X_test)[:, 1]
leaky_auc = roc_auc_score(yte_grp, leaky_scores)

honest_auc = after_result["roc_auc"]

leak_audit = pd.DataFrame([
    {"feature_set": "honest (5 features)", "roc_auc": honest_auc},
    {"feature_set": "with trend_pct (leak)", "roc_auc": leaky_auc},
])
leak_audit


**Audit conclusion:** `# FILL AFTER RUN` — confirm the leaky row's AUC is clearly higher
(should trend toward ~1.0) and restate: `trend_pct` stays excluded from the kept model. The honest
AUC used everywhere else in this repo is the "honest (5 features)" row above, from the grouped
split.

**Time-window note (not fully audited here):** the label being read from the same window as the
features (rather than a true future window) is a design limitation carried over from Weeks 3-5,
not something this audit fixes — a stronger version of this project would rebuild the label as
"features from March → outcome in April," which is flagged as future work rather than done here,
since it would change the whole feature/label pipeline built in earlier weeks.


## 4) Claim Rewrite

Pulling forward claims made in `w03`-`w05` and rewriting any that go further than the evidence
supports, using safe language (**observed**, **measured**, **directional**, **decision-support**)
throughout. Fill in the actual original wording from your own notebooks on the left before
finalizing the rewrite on the right.


| Where it was said | Original claim | Rewritten, safe-language version |
|---|---|---|
| w03, feature associations | `[paste your original wording]` | "We **observed** an association between `[feature]` and the proxy decline label in the March slice; this is **directional**, not causal, and not yet tested against a future outcome." |
| w04, baseline rule | `[paste your original wording]` | "The baseline queue is a **decision-support** ranking — it orders pages for human review by two **measured** signals (staleness, CTR gap vs. tier); it does not claim a refresh will recover traffic." |
| w05, model vs baseline | `[paste your original wording]` | "The model's precision@50 was **measured** as higher than the baseline's on this held-out, client-grouped test split; this is a comparison on one proxy label and one month, not a general claim about all content." |
| w05, feature importance | `[paste your original wording]` | "Permutation importance **observed** `[feature]` as the strongest driver of the model's ranking on this split; this describes what the model leaned on, not a proven causal driver of decline." |

`# FILL AFTER RUN` — add any other claim from earlier notebooks that reads as more certain than
the evidence supports (watch especially for "predicts," "causes," "proves," or "will recover" —
those are the words section 14 of the lane guide flags as not public-safe) and rewrite it the same
way.


## 5) Self-Check

- [ ] Two paper findings named, each with a constructive methodology question — section 1
      (requires your own read of the paper; not something filled in on your behalf).
- [ ] Model re-run under an honest split with a real before/after comparison — section 2.
- [ ] At least one split improvement shown (grouped-by-client here; time-aware noted as future
      work, not silently skipped).
- [ ] Leakage audit: checklist answered for all five features, plus the deliberate-leak metric
      re-run — section 3.
- [ ] Failure/leak examples are concrete (actual overlap counts, actual AUC numbers), not just
      described in the abstract.
- [ ] Claims rewritten in safe language (observed / measured / directional / decision-support)
      everywhere, including the ones carried over from w03-w05 — section 4.
- [ ] No claim states or implies a Google algorithm factor, a proven causal refresh effect, or a
      guarantee — matches section 14 of the lane guide.
